In [4]:
pip install xgboost imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 17.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/processed/telco_churn_cleaned.csv')

# ---- FEATURE ENGINEERING ----
# Drop customerID — not a feature
df = df.drop('customerID', axis=1)

# Encode binary columns
binary_cols = ['Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling']
for col in binary_cols:
    df[col] = (df[col] == 'Yes').astype(int)

# Encode MultipleLines
df['MultipleLines'] = df['MultipleLines'].map(
    {'Yes': 1, 'No': 0, 'No phone service': 0})

# Encode internet service columns
internet_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in internet_cols:
    df[col] = df[col].map(
        {'Yes': 1, 'No': 0, 'No internet service': 0})

# One hot encode remaining categoricals
df = pd.get_dummies(df, columns=['InternetService', 'Contract',
                                  'PaymentMethod'], drop_first=True)

# Drop tenure_group if exists
if 'tenure_group' in df.columns:
    df = df.drop('tenure_group', axis=1)

# Gender encode
df['gender'] = (df['gender'] == 'Male').astype(int)

print("Features shape:", df.shape)
print("Churn rate:", df['Churn'].mean())

# ---- TRAIN TEST SPLIT ----
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTrain size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")

# ---- HANDLE IMBALANCE WITH SMOTE ----
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"\nAfter SMOTE - Train size: {X_train_sm.shape[0]}")
print(f"Churn distribution after SMOTE: {y_train_sm.value_counts().to_dict()}")

# ---- SCALE FEATURES ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

# ---- TRAIN THREE MODELS ----
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train_sm)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:,1]
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'auc': auc
    }
    print(f"\n{name} — AUC: {auc:.4f}")
    print(classification_report(y_test, y_pred))

Features shape: (7032, 24)
Churn rate: 0.26578498293515357

Train size: 5625
Test size: 1407

After SMOTE - Train size: 8260
Churn distribution after SMOTE: {0: 4130, 1: 4130}

Logistic Regression — AUC: 0.8105
              precision    recall  f1-score   support

           0       0.86      0.81      0.83      1033
           1       0.55      0.63      0.59       374

    accuracy                           0.76      1407
   macro avg       0.70      0.72      0.71      1407
weighted avg       0.78      0.76      0.77      1407


Random Forest — AUC: 0.8057
              precision    recall  f1-score   support

           0       0.85      0.82      0.84      1033
           1       0.56      0.60      0.58       374

    accuracy                           0.77      1407
   macro avg       0.70      0.71      0.71      1407
weighted avg       0.77      0.77      0.77      1407


XGBoost — AUC: 0.8002
              precision    recall  f1-score   support

           0       0.86     